<a href="https://colab.research.google.com/github/NVKatyayani/katchuram/blob/main/AI_Threat_Intelligence_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q sentence-transformers gradio
import os, re, glob
import numpy as np
print("ready")

ready


In [2]:
# STEP 1: Create the base knowledge directory and custom uploads folder
os.makedirs("knowledge/threats", exist_ok=True)
os.makedirs("knowledge/vulnerabilities", exist_ok=True)
os.makedirs("knowledge/custom_uploads", exist_ok=True)

# STEP 2: Default threat notes (OKF format)
THREAT_NOTES = {
"knowledge/threats/ransomware_lockbit.md": """---
type: ThreatIntel
title: LockBit Ransomware
---
# LockBit Ransomware
## Indicators of Compromise
Associated file hashes include malicious payloads dropping ransom notes. Look for unusual PowerShell script execution.
## Mitigation
Isolate affected endpoints immediately, disable compromised user accounts, and restore from offline immutable backups.""",

"knowledge/vulnerabilities/cve_2026_xyz.md": """---
type: Vulnerability
title: Critical Remote Code Execution CVE-2026-XYZ
---
# Critical Remote Code Execution CVE-2026-XYZ
## Affected Systems
Web servers running API Gateway version 4.2 or lower.
## Remediation
Apply patch version 4.3 immediately or restrict external access to port 8443 via firewall rules.""",
}

for path, text in THREAT_NOTES.items():
    open(path, "w").write(text)
    print("saved default note:", path)

saved default note: knowledge/threats/ransomware_lockbit.md
saved default note: knowledge/vulnerabilities/cve_2026_xyz.md


In [3]:
from google.colab import files

print("Please select your threat intelligence files (.txt or .md) to upload:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Save uploaded files into the custom uploads folder
    dest_path = os.path.join("knowledge/custom_uploads", filename)
    os.rename(filename, dest_path)
    print(f"Successfully uploaded and stored: {dest_path}")

Please select your threat intelligence files (.txt or .md) to upload:


Saving phishing.txt to phishing.txt
Successfully uploaded and stored: knowledge/custom_uploads/phishing.txt


In [4]:
files_list = [f for f in sorted(glob.glob("knowledge/**/*.md", recursive=True))
              if os.path.basename(f) not in ("index.md", "log.md")]

# Also include any uploaded .txt files by converting them or treating them as text
txt_files = glob.glob("knowledge/**/*.txt", recursive=True)

problems = 0
for f in files_list:
    text = open(f, encoding="utf-8", errors="ignore").read()
    missing = []
    if "type:" not in text: missing.append("a type: line")
    if not re.search(r"^# .+", text, re.M): missing.append("a # Title")
    if "## " not in text: missing.append("at least one ## section")
    if missing:
        problems += 1
        print("PROBLEM", f, "is missing", ", ".join(missing))
    else:
        print("OK     ", f)

index = "---\ntype: Index\title: Threat Intel Index\n---\n# Threat Intel Index\n" + "".join(
    f"- {f.replace('knowledge/', '')}\n" for f in files_list + txt_files)
open("knowledge/index.md", "w").write(index)
print("\nAll documents verified & index updated!")

OK      knowledge/threats/ransomware_lockbit.md
OK      knowledge/vulnerabilities/cve_2026_xyz.md

All documents verified & index updated!


In [5]:
from sentence_transformers import SentenceTransformer

def load_pieces():
    pieces = []
    # Load Markdown files
    for path in sorted(glob.glob("knowledge/**/*.md", recursive=True)):
        if os.path.basename(path) in ("index.md", "log.md"):
            continue
        text = open(path, encoding="utf-8", errors="ignore").read()
        title_match = re.search(r"^# (.+)$", text, re.M)
        title = title_match.group(1) if title_match else os.path.basename(path)

        parts = text.split("## ")
        if len(parts) > 1:
            for part in parts[1:]:
                pieces.append({
                    "file": path,
                    "text": f"{title}: {part.strip()}",
                    "status": "draft" if "status: draft" in text else ""
                })
        else:
            pieces.append({"file": path, "text": text, "status": ""})

    # Load plain text (.txt) files if any were uploaded
    for path in sorted(glob.glob("knowledge/**/*.txt", recursive=True)):
        text = open(path, encoding="utf-8", errors="ignore").read()
        pieces.append({"file": path, "text": text, "status": ""})

    return pieces

PIECES = load_pieces()
model = SentenceTransformer("all-MiniLM-L6-v2")
VECTORS = model.encode([p["text"] for p in PIECES], normalize_embeddings=True)

def find(question):
    q = model.encode([question], normalize_embeddings=True)[0]
    scores = VECTORS @ q
    i = int(np.argmax(scores))
    return float(scores[i]), PIECES[i]

print(f"Total intelligence segments indexed: {len(PIECES)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total intelligence segments indexed: 5


In [6]:
LIMIT = 0.30
USE_MODEL = True   # Set to True to let the small AI write answers, False to show raw text notes
_chat = None

def write_answer(question, note):
    global _chat
    if _chat is None:
        from transformers import pipeline
        _chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")
    messages = [
        {"role": "system", "content": "You are a cybersecurity threat intelligence analyst. Answer strictly using the provided threat notes in 1-2 concise sentences. If unknown, say: I don't have intelligence on this threat."},
        {"role": "user", "content": f"Threat Notes: {note}\nSecurity Query: {question}"}]
    out = _chat(messages, max_new_tokens=80, do_sample=False)[0]["generated_text"]
    return out[-1]["content"].strip()

def agent(question, show=True):
    score, best = find(question)
    if score < LIMIT:
        file, answer = None, "I could not find matching threat intelligence in my database."
    else:
        file = best["file"]
        answer = write_answer(question, best["text"]) if USE_MODEL else best["text"]
    if show:
        print("QUERY:", question)
        print("SOURCE FILE:", file, f"(relevance score {score:.2f})")
        print("INTELLIGENCE BRIEF:", answer, "\n")
    return file

# Quick test
agent("How do I mitigate LockBit ransomware?")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


QUERY: How do I mitigate LockBit ransomware?
SOURCE FILE: knowledge/threats/ransomware_lockbit.md (relevance score 0.81)
INTELLIGENCE BRIEF: To mitigate LockBit ransomware, isolate affected endpoints, disable compromised user accounts, and restore from offline immutable backups. 



'knowledge/threats/ransomware_lockbit.md'

In [10]:
import gradio as gr

def ask_agent(question):
    if not question.strip():
        return "", "", "Please type a security query."
    score, best = find(question)
    if score < LIMIT:
        return "None", f"{score:.2f}", "No matching threat intelligence found."
    answer = write_answer(question, best["text"]) if USE_MODEL else best["text"]
    return f"{best['file']}\n\n{best['text']}", f"{score:.2f}", answer

with gr.Blocks(title="AI Threat Intelligence Assistant") as demo:
    gr.Markdown("# 🛡️ AI Threat Intelligence Assistant\nQuery your internal security documents, CVE playbooks, and uploaded custom files.")
    q = gr.Textbox(label="Security Query / IOC Question", placeholder="Ask something about your threat notes...")
    btn = gr.Button("Query Intelligence Database", variant="primary")
    found = gr.Textbox(label="Retrieved Source Document", lines=4)
    score_box = gr.Textbox(label="Confidence Score")
    ans = gr.Textbox(label="Synthesized Response", lines=4)

    # Ikkada 4 example questions add chesamu
    gr.Examples([
        "How do I mitigate LockBit ransomware?",
        "What are the affected systems for CVE-2026-XYZ?",
        "What are the indicators of compromise for LockBit?",
        "How do we patch CVE-2026-XYZ?"
    ], inputs=q)

    btn.click(ask_agent, q, [found, score_box, ans])
    q.submit(ask_agent, q, [found, score_box, ans])

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://317b9ce007f8f1caaf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
